In [ ]:
import math
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from utils import load_video, play_video
from IPython.display import clear_output

In [ ]:
video = load_video("data/calib/0.mp4", gray=True)
print(video.shape)

In [ ]:
calib_data = np.load("data/calib/calib_data.npz")
calib_mtx = calib_data["mtx"]
calib_dist = calib_data["dist"]
calib_new_mtx = calib_data["new_mtx"]
print(calib_new_mtx)

In [ ]:
play_video(video[::5])

In [ ]:
FAST_THRESH = 30
FB_ERR_THRESH = 1.0  # forward-backward error

REDETECT_THRESHOLD = 50
REDETECT_MIN_DIST = 10
REDETECT_INTERVAL = 20

vis_frames = []
pt_pairs = []

fast = cv2.FastFeatureDetector_create(threshold=FAST_THRESH, nonmaxSuppression=True)

pts_1 = np.empty((0, 2), dtype=np.float32)
pts_2 = np.empty((0, 2), dtype=np.float32)

for curr_idx in range(1, video.shape[0]):
    img_1 = video[curr_idx - 1]
    img_2 = video[curr_idx]

    # redetect at regular intervals or if too few points are tracked
    pts_1 = pts_2.tolist()  # carry over tracked points
    # if (curr_idx - 1) % REDETECT_INTERVAL == 0 or len(pts_1) < REDETECT_THRESHOLD:
    if curr_idx == 1:
        print("Redetecting features...")
        new_pts = np.array([p.pt for p in fast.detect(img_2, None)], dtype=np.float32)
        for p in new_pts:
            if not any(np.linalg.norm(p - q) < REDETECT_MIN_DIST for q in pts_1):
                pts_1.append(p)
    pts_1 = np.array(pts_1, dtype=np.float32)

    forw_pts, forw_status, _ = cv2.calcOpticalFlowPyrLK(img_1, img_2, pts_1, None)
    back_pts, back_status, _ = cv2.calcOpticalFlowPyrLK(img_2, img_1, forw_pts, None)

    fb_err = np.linalg.norm(back_pts - pts_1, axis=1)
    good = (
        (forw_status.flatten() == 1)
        & (back_status.flatten() == 1)
        & (fb_err < FB_ERR_THRESH)
    )

    # filter points where tracking failed or went out of bounds
    filtered_pts_1 = []
    filtered_pts_2 = []
    for i in range(len(good)):
        if good[i]:
            x, y = forw_pts[i]
            h, w = img_2.shape
            if 0 <= x < w and 0 <= y < h:
                filtered_pts_1.append(pts_1[i])
                filtered_pts_2.append(forw_pts[i])
    pts_1 = np.array(filtered_pts_1, dtype=np.float32)
    pts_2 = np.array(filtered_pts_2, dtype=np.float32)
    pt_pairs.append((pts_1.copy(), pts_2.copy()))

    # visualize tracked points
    vis_frame = img_2.copy()
    for p in pts_2:
        x, y = p
        cv2.circle(vis_frame, (int(x), int(y)), 3, (255,), -1)
    vis_frames.append(vis_frame)

In [ ]:
# for each pair, print avg delta between each corresponding point
for i, (pts_1, pts_2) in enumerate(pt_pairs):
    deltas = pts_2 - pts_1
    dists = np.linalg.norm(deltas, axis=1)
    avg_dist = np.mean(dists)
    print(f"Frame {i} -> {i+1}: Avg point movement = {avg_dist:.2f} pixels")

In [ ]:
trans = []
rots = []
for i in range(len(pt_pairs)):
    pts_1, pts_2 = pt_pairs[i]
    E, mask = cv2.findEssentialMat(
        pts_1, pts_2, calib_new_mtx, method=cv2.RANSAC, prob=0.999, threshold=1.0
    )
    _, R, t, mask = cv2.recoverPose(E, pts_1, pts_2, calib_new_mtx)
    rots.append(R)
    trans.append(t)

In [ ]:
# rot_0 -> (frame 0 -> 1)
# rot_1 -> (frame 1 -> 2)
# rot_2 -> (frame 2 -> 3)
# rot_3 -> (frame 3 -> 4)
# rot_4 -> (frame 4 -> 5)

In [ ]:
from scipy.spatial.transform import Rotation as RS

q = RS.from_matrix(rots).as_quat()   # (x,y,z,w)
for i in range(1, len(q)):
    if np.dot(q[i-1], q[i]) < 0:
        q[i] = -q[i]                 # same rotation, continuous sign

# yaw-pitch-roll = rotations about z, y, x
ypr = RS.from_quat(q).as_euler('xyz', degrees=True)
roll = ypr[:, 0]
pitch = ypr[:, 1]
yaw = ypr[:, 2]
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axs[0].plot(roll, label='Roll', color='r')
axs[0].set_ylabel('Degrees')
axs[0].legend()
axs[1].plot(pitch, label='Pitch', color='g')
axs[1].set_ylabel('Degrees')
axs[1].legend()
axs[2].plot(yaw, label='Yaw', color='b')
axs[2].set_ylabel('Degrees')
axs[2].set_xlabel('Keyframe Index')
axs[2].legend()
plt.suptitle('Estimated Orientation (Yaw-Pitch-Roll) Over Time')
plt.show()

In [ ]:
# plot translation directions
trans = np.array(trans).squeeze()  # (N, 3)
trans_dirs = trans / np.linalg.norm(trans, axis=1, keepdims=True)  # normalize
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.quiver(
    np.zeros(trans_dirs.shape[0]),
    np.zeros(trans_dirs.shape[0]),
    np.zeros(trans_dirs.shape[0]),
    trans_dirs[:, 0],
    trans_dirs[:, 1],
    trans_dirs[:, 2],
    length=0.1, normalize=True,
)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Estimated Translation Directions Over Time')
plt.show()